# Table Processing in RAG: Failure → Description Remedy
### Why linear text extraction destroys row/column structure, and how to fix it

In [1]:
!pip install langchain langchain-community langchain-openai langchain-pinecone pinecone langchain-text-splitters pypdf pdfplumber python-dotenv -q


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import time
from dotenv import load_dotenv
load_dotenv()

import pdfplumber
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

PDF_PATH = "Q4 2025-26 Fact Sheet.pdf"
INDEX_NAME = "mmrag-openai"
NAMESPACE = "factsheet"

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

C:\Users\shiva\AppData\Local\Temp\ipykernel_11312\2509548087.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


C:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Naive text-only baseline
`PyPDFLoader` reads this deck's text layer linearly, left to right, line by line. That alone doesn't scramble every table — each row is usually stored as one contiguous run of text, numbers already in the right order. The failure shows up when a table's *header* itself doesn't fit on one line: page 11's "Growth by Market" table has 11 numeric columns per row — quarterly Q4 FY25 / Q3 FY26 / Q4 FY26 figures, then four growth metrics whose header text wraps across three stacked lines ("Q-o-Q" / "CC Growth", "Y-o-Y" / "CC Growth", "Q-o-Q" / "INR Growth", "Y-o-Y" / "INR Growth"), then annual FY25/FY26 figures and two more growth columns. Each row's 11 numbers stay in order, but nothing in the flattened text pins the 6th number to its correct (wrapped, three-line) header instead of, say, the 10th.

In [3]:
pc = Pinecone()
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,  # text-embedding-3-small output size
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)

text_docs = PyPDFLoader(PDF_PATH).load()
text_chunks = splitter.split_documents(text_docs)
for d in text_chunks:
    d.metadata["chunk_type"] = "text"

vs = PineconeVectorStore.from_documents(text_chunks, embedding=embeddings, index_name=INDEX_NAME, namespace=NAMESPACE)
print(f"Baseline: {len(text_chunks)} text-only chunks upserted into '{INDEX_NAME}/{NAMESPACE}'")

Baseline: 38 text-only chunks upserted into 'mmrag-openai/factsheet'


## Step 2: Ask a table-specific question
Page 11's "Growth by Market" table lists India's Q-o-Q INR Growth at 1.8% for Q4 FY26 — one number out of eleven on India's row, flanked by similarly-named columns (Q-o-Q CC Growth, Y-o-Y INR Growth) and, ten columns later on that same row, an *annual* "YoY CC Growth" / "YoY INR Growth" pair that reads almost the same in plain text. Nothing marks which of India's eleven numbers is the one actually asked for.

In [4]:
RAG_PROMPT = """Answer the question using only the following context. If the context does not contain the answer, say so plainly instead of guessing.

Context:
{context}

Question: {question}
Answer:"""

def answer_from_index(question, k=4):
    hits = vs.similarity_search(question, k=k, namespace=NAMESPACE)
    context = "\n\n".join(f"[{h.metadata.get('chunk_type')}] {h.page_content}" for h in hits)
    response = llm.invoke(RAG_PROMPT.format(context=context, question=question)).content.strip()
    return hits, response

question = "What was the Q-o-Q INR Growth (%) for India in TCS's Growth by Market table (Q4 FY26)?"
hits, response = answer_from_index(question)
print("Retrieved chunk_types:", [h.metadata.get("chunk_type") for h in hits])
print("\nAnswer:\n", response)

Retrieved chunk_types: ['text', 'text', 'text', 'text']

Answer:
 The Q-o-Q INR Growth (%) for India in TCS's Growth by Market table (Q4 FY26) is -28.6%.


## Failure: structure destroyed, not invisibility
This is a different failure from Notebook 1's missing image. The correct answer (1.8%) is physically present in the retrieved text — the model has every digit it needs. What's missing is the alignment between each digit and its column header, so the model latches onto a different column from the same row instead — India's *annual* Y-o-Y CC/INR Growth (-28.6% / -28.5%), ten columns away from the quarterly Q-o-Q INR Growth actually asked about. Captioning an image and untangling a table are different problems that happen to share the same fix.

## Step 3: Extract the table with pdfplumber
Unlike `PyPDFLoader`'s linear text stream, `pdfplumber` detects the table's actual grid — rows and columns as pdfplumber sees the lines/positions on the page — and returns it as a nested list that still reflects that structure.

In [5]:
TABLE_PAGE_INDEX = 10  # 0-indexed -> human page 11, "Growth by Market"

def extract_grid_from_words(page, x_tolerance=10, row_gap_tol=7, col_gap_tol=35):
    """Fallback for tables pdfplumber's grid detectors can't see.

    This page has no ruling lines (default line-based `extract_tables()`
    returns zero tables), and its header wraps across THREE stacked lines
    ("Q-o-Q" over "CC Growth", etc.), while row labels like "North America"
    and negative figures like "- 23.0" are themselves split across multiple
    word-boxes. A naive per-line read has no reliable way to tell which of
    a row's 11 trailing numbers lines up with which wrapped header — that's
    exactly what makes the baseline answer wrong below.

    Reconstruct the grid in two passes over each word's (top, x0) position:
    1. Cluster words into ROWS by shared `top` (a gap tolerance merges the
       header's 3 stacked lines into one row, while each data row — spaced
       much further from its neighbours — stays on its own).
    2. Within each row, cluster words into COLUMNS by x-proximity (a gap
       tolerance merges multi-word labels like "North"+"America" and
       wrapped header fragments like "Q-o-Q"+"CC"+"Growth" into one cell,
       ordering sub-words top-to-bottom then left-to-right so "Q-o-Q CC
       Growth" comes out in the right order instead of "CC Q-o-Q Growth").
    A final pass reattaches a minus sign pdfplumber split into its own
    token (e.g. '-', '23.0' -> '-23.0').
    """
    words = sorted(page.extract_words(x_tolerance=x_tolerance), key=lambda w: w["top"])
    rows, current, current_top = [], [], None
    for w in words:
        if current and (w["top"] - current_top) > row_gap_tol:
            rows.append(current)
            current = []
        current.append(w)
        current_top = w["top"]
    if current:
        rows.append(current)

    grid = []
    for row_words in rows:
        row_words_sorted = sorted(row_words, key=lambda w: w["x0"])
        cols, current_col, current_x = [], [], None
        for w in row_words_sorted:
            if current_col and (w["x0"] - current_x) > col_gap_tol:
                cols.append(current_col)
                current_col = []
            current_col.append(w)
            current_x = w["x0"]
        if current_col:
            cols.append(current_col)
        cells = [
            " ".join(w["text"] for w in sorted(col, key=lambda w: (w["top"], w["x0"])))
            for col in cols
        ]

        merged = []
        i = 0
        while i < len(cells):
            if cells[i] == "-" and i + 1 < len(cells):
                nxt = cells[i + 1].replace(",", "")
                try:
                    float(nxt)
                    merged.append("-" + cells[i + 1])
                    i += 2
                    continue
                except ValueError:
                    pass
            merged.append(cells[i])
            i += 1
        grid.append(merged)
    return grid

with pdfplumber.open(PDF_PATH) as pdf:
    page = pdf.pages[TABLE_PAGE_INDEX]
    tables = page.extract_tables()
    table = tables[0] if tables else extract_grid_from_words(page)

for row in table:
    print(row)

['Growth', 'by Market']
['Geography', '(%)', 'Q4 FY25', 'Q3 FY26', 'Q4 FY26', 'Q-o-Q CC Growth', 'Y-o-Y CC Growth', 'Q-o-Q INR Growth', 'Y-o-Y INR Growth', 'FY25', 'FY26', 'YoY CC Growth', 'YoY INR Growth']
['Americas']
['North America', '48.2', '48.5', '48.5', '1.4', '2.5', '5.4', '10.5', '48.2', '48.6', '0.2', '5.5']
['Latin America', '1.8', '2.0', '1.9', '- 6.9', '- 2.9', '- 1.9', '12.8', '1.9', '1.9', '0.9', '8.4']
['Europe']
['UK', '16.8', '16.9', '17.2', '2.4', '- 1.2', '7.3', '12.3', '16.8', '17.4', '-1.9', '8.1']
['Continental', 'Europe', '14.3', '15.6', '15.6', '1.0', '1.0', '5.3', '19.2', '14.3', '15.4', '-0.9', '12.4']
['Asia Pacific', '8.1', '8.3', '8.3', '- 0.5', '0.4', '5.7', '13.1', '8.0', '8.3', '2.4', '9.5']
['India', '8.4', '6.1', '6.0', '1.7', '- 23.0', '1.8', '- 22.8', '8.6', '5.9', '-28.6', '-28.5']
['MEA', '2.4', '2.6', '2.5', '0.4', '7.8', '4.9', '18.6', '2.2', '2.5', '9.5', '16.7']
['Total', '100.0', '100.0', '100.0', '1.2', '-0.6', '5.4', '9.6', '100.0', '100.0

## Step 4: Describe the table row-by-row in natural language
The grid is rendered as pipe-joined rows and handed to the LLM with an instruction to rewrite it as explicit "metric / period / value" sentences — spelling out exactly the alignment a linear text extraction throws away.

In [6]:
TABLE_DESC_PROMPT = """The following is a financial table extracted as a grid of rows, with `None` marking empty cells. 
Rewrite it as a series of explicit sentences, one per metric per period, in the exact form:
"<metric> was <value> in <period>."
Use the header row to identify each period. Do not skip any row or number.

Table title: {title}

Grid:
{grid}

Description:"""

def describe_table(table, title):
    grid_text = "\n".join(" | ".join("" if cell is None else str(cell) for cell in row) for row in table)
    return llm.invoke(TABLE_DESC_PROMPT.format(title=title, grid=grid_text)).content.strip()

table_description = describe_table(table, title="Growth by Market (Geography)")
print(table_description)

1. North America was 48.2% in Q4 FY25.
2. North America was 48.5% in Q3 FY26.
3. North America was 48.5% in Q4 FY26.
4. North America had a Q-o-Q CC Growth of 1.4% in Q4 FY26.
5. North America had a Y-o-Y CC Growth of 2.5% in Q4 FY26.
6. North America had a Q-o-Q INR Growth of 5.4% in Q4 FY26.
7. North America had a Y-o-Y INR Growth of 10.5% in Q4 FY26.
8. North America was 48.2% in FY25.
9. North America was 48.6% in FY26.
10. North America had a YoY CC Growth of 0.2% in FY26.
11. North America had a YoY INR Growth of 5.5% in FY26.

12. Latin America was 1.8% in Q4 FY25.
13. Latin America was 2.0% in Q3 FY26.
14. Latin America was 1.9% in Q4 FY26.
15. Latin America had a Q-o-Q CC Growth of -6.9% in Q4 FY26.
16. Latin America had a Y-o-Y CC Growth of -2.9% in Q4 FY26.
17. Latin America had a Q-o-Q INR Growth of -1.9% in Q4 FY26.
18. Latin America had a Y-o-Y INR Growth of 12.8% in Q4 FY26.
19. Latin America was 1.9% in FY25.
20. Latin America was 1.9% in FY26.
21. Latin America had a Y

## Step 5: Embed the description into the SAME index
This table has 9 rows × 11 columns of description — noticeably bigger than a single small income-statement table. Embedding the *whole* description as one chunk would average every country's facts into one vector, diluting India's specific numbers enough that the chunk stops competing with the baseline's flattened text at retrieval time. Splitting the description into one chunk per row (the LLM's output already separates rows with blank lines) keeps each row's facts sharp enough to actually win at `k=4`.

In [7]:
# One chunk per row instead of one chunk for the whole table — see Step 5 above.
table_paragraphs = [p.strip() for p in table_description.split("\n\n") if p.strip()]
table_docs = [
    Document(
        page_content=p,
        metadata={"chunk_type": "table_desc", "page": TABLE_PAGE_INDEX + 1, "table_title": "Growth by Market"},
    )
    for p in table_paragraphs
]
vs.add_documents(table_docs, namespace=NAMESPACE)
print(f"Added {len(table_docs)} table-description chunks into '{INDEX_NAME}/{NAMESPACE}'")

Added 8 table-description chunks into 'mmrag-openai/factsheet'


## Step 6: Re-run the same question

In [8]:
hits, response = answer_from_index(question)
print("Retrieved chunk_types:", [h.metadata.get("chunk_type") for h in hits])
print("\nAnswer:\n", response)

Retrieved chunk_types: ['table_desc', 'table_desc', 'text', 'text']

Answer:
 The Q-o-Q INR Growth for India in TCS's Growth by Market table (Q4 FY26) was 1.8%.


## Wrap-up: two faces of the same fix
Notebook 1's images and this notebook's tables fail for opposite-sounding reasons — one is missing, one is present but scrambled — yet the remedy is identical: describe the non-text-native content in plain language, embed that description like any other chunk. Once described, both problems become the same ordinary text-RAG problem downstream.

## Try it yourself
1. Extract and describe the domain table (page 12) or the two-stacked-tables "COR – SG&A Details" page (page 18) the same way, and ask a question specific to one of their rows.
2. If a table's borders aren't detected cleanly, retry with `page.extract_tables(table_settings={"vertical_strategy": "text", "horizontal_strategy": "text"})` — pdfplumber's default line-based detection can miss borderless tables, though for badly letter-spaced or wrapped-header pages (like this notebook's page 11) even that can fall short, which is why this notebook falls back to reconstructing the grid from raw word positions instead.
3. Compare the baseline's raw flattened text for page 11 against the structured `pdfplumber` grid side by side to see exactly where the alignment breaks.

**Cleanup:** this notebook writes into the shared `mmrag-openai` index under the `factsheet` namespace only (Notebook 1 uses the same index under `uan`). Delete just this namespace when done:
```python
pc.Index(INDEX_NAME).delete(delete_all=True, namespace=NAMESPACE)
```